## MCSBT_MIDTERM_ASSIGNMENT :: ROI IDEA to Word2Vec Modeling with Spark MLLib

### Install and initialize PySpark

In [4]:
#!pip install pyspark

In [5]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [6]:
spark = SparkSession.builder.getOrCreate()

In [7]:
print("Using Apache Spark Version", spark.version)

Using Apache Spark Version 4.0.4


### Process Crunchbase Open Data Map in Spark DataFrame

#### Read the csv file into Spark DataFrame

In [10]:
cb = spark.read.options(inferSchema='True',delimiter=',',header='True') \
          .csv("/content/sample_data/cb_odm_092419.csv")
cb.count()

64341

#### Filter out empty descriptions from the dataframe

In [11]:
cb = cb.select('crunchbase_uuid','name','short_description') \
       .filter('short_description != ""')
cb.count()

64316

#### Create a new column, "inputText", for downstream NLP processes

In [ ]:
df = cb.withColumn('inputText', F.col('short_description'))

In [ ]:
df.show(5)

+--------------------+--------+--------------------+--------------------+
|     crunchbase_uuid|    name|   short_description|           inputText|
+--------------------+--------+--------------------+--------------------+
|e1393508-30ea-8a3...|Wetpaint|Wetpaint offers a...|Wetpaint offers a...|
|bf4d7b0e-b34d-2fd...|    Zoho|Zoho offers a sui...|Zoho offers a sui...|
|5f2b40b8-d1b3-d32...|    Digg|Digg Inc. operate...|Digg Inc. operate...|
|df662812-7f97-0b4...|Facebook|Facebook is an on...|Facebook is an on...|
|b08efc27-da40-505...|   Accel|Accel is an early...|Accel is an early...|
+--------------------+--------+--------------------+--------------------+
only showing top 5 rows



### Train a Word2Vec Model on Crunchbase Open Data Map

#### Apply Regex Tokenizer and StopWord Remover

In [ ]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, Word2Vec

In [ ]:
# regular expression tokenizer to tokenize inputText into individual tokens (words)
regextok = RegexTokenizer(gaps = False, pattern = '\w+',
                          inputCol = 'inputText', outputCol = 'tokens')

<>:2: SyntaxWarning: invalid escape sequence '\w'
<>:2: SyntaxWarning: invalid escape sequence '\w'
/var/folders/_w/n0040jg122g3jhjxrmwpkt2w0000gp/T/ipykernel_83385/2152546088.py:2: SyntaxWarning: invalid escape sequence '\w'
  regextok = RegexTokenizer(gaps = False, pattern = '\w+',


In [ ]:
# StopWordsRemover to remove stopwords in the list of tokens
stopwrmv = StopWordsRemover(inputCol = 'tokens', outputCol = 'tokens_sw_removed')

In [ ]:
df = regextok.transform(df)
df = stopwrmv.transform(df)
df.show(3)

+--------------------+--------+--------------------+--------------------+--------------------+--------------------+
|     crunchbase_uuid|    name|   short_description|           inputText|              tokens|   tokens_sw_removed|
+--------------------+--------+--------------------+--------------------+--------------------+--------------------+
|e1393508-30ea-8a3...|Wetpaint|Wetpaint offers a...|Wetpaint offers a...|[wetpaint, offers...|[wetpaint, offers...|
|bf4d7b0e-b34d-2fd...|    Zoho|Zoho offers a sui...|Zoho offers a sui...|[zoho, offers, a,...|[zoho, offers, su...|
|5f2b40b8-d1b3-d32...|    Digg|Digg Inc. operate...|Digg Inc. operate...|[digg, inc, opera...|[digg, inc, opera...|
+--------------------+--------+--------------------+--------------------+--------------------+--------------------+
only showing top 3 rows



#### Train Word2Vec Model on the Crunchbase Dataset

In [ ]:
word2vec = Word2Vec(vectorSize = 100, minCount = 5, inputCol = 'tokens_sw_removed', outputCol = 'wordvectors')

In [ ]:
model = word2vec.fit(df)

24/07/06 00:31:24 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


#### Search for synonyms in the word2vec model based on similarity

In [ ]:
synonyms = model.findSynonyms("microsoft", 20)
synonyms.show()

+----------+------------------+
|      word|        similarity|
+----------+------------------+
|       nav| 0.767176628112793|
|        ax|0.7506062388420105|
|     azure| 0.745636522769928|
|sharepoint|0.7416457533836365|
|    oracle|0.7291754484176636|
|  dynamics|0.7094357013702393|
|       ibm|0.7081155776977539|
|  netsuite|0.6768943071365356|
|       aws|0.6698634624481201|
|       sap|0.6531175374984741|
|salesforce|0.6418511867523193|
|       isv| 0.634341299533844|
|       365|0.6301283240318298|
| openstack| 0.623803973197937|
| acumatica|0.6153972148895264|
|       soa|0.6019008159637451|
|    epicor|0.5938819646835327|
|       sql|0.5879281163215637|
|  navision|0.5870295166969299|
| filemaker| 0.584995687007904|
+----------+------------------+



### Query the text using word2vec model

#### Reload the original crunchbase data and create an inputText column

In [ ]:
df = cb.withColumn('inputText', F.col('short_description'))

#### Apply the regex tokenizer, stopword remover, and word2vec model

In [ ]:
df = regextok.transform(df)
df = stopwrmv.transform(df)
df = model.transform(df)

In [ ]:
df.select('crunchbase_uuid','name','wordvectors').show(5)

+--------------------+--------+--------------------+
|     crunchbase_uuid|    name|         wordvectors|
+--------------------+--------+--------------------+
|e1393508-30ea-8a3...|Wetpaint|[0.10230748293300...|
|bf4d7b0e-b34d-2fd...|    Zoho|[0.13701602417443...|
|5f2b40b8-d1b3-d32...|    Digg|[0.07723724161489...|
|df662812-7f97-0b4...|Facebook|[0.19956633606925...|
|b08efc27-da40-505...|   Accel|[0.02619387247514...|
+--------------------+--------+--------------------+
only showing top 5 rows



In [ ]:
df.count()

684280

#### Selecting 30,000 entries for querying

In [ ]:
chunks = df.select('crunchbase_uuid','name','wordvectors').limit(30000).collect()

In [ ]:
chunks[0]

Row(crunchbase_uuid='e1393508-30ea-8a36-3f96-dd3226033abd', name='Wetpaint', wordvectors=DenseVector([0.1023, 0.0621, -0.051, 0.1017, -0.0687, -0.0639, -0.1182, -0.0996, -0.0093, -0.0358, 0.0208, -0.0773, 0.1145, 0.1586, -0.1143, -0.133, -0.0027, 0.0486, 0.0359, 0.0357, -0.0306, -0.1151, 0.0167, -0.0205, -0.0992, 0.0278, 0.066, 0.0148, 0.0447, -0.1047, 0.1446, -0.0399, 0.0164, -0.0747, -0.1977, 0.0114, -0.02, -0.0285, -0.063, 0.1241, -0.1463, 0.1822, 0.0687, -0.1114, -0.0137, -0.198, 0.0667, 0.0915, 0.0491, -0.2335, 0.1199, -0.0297, -0.0037, 0.0819, -0.0908, -0.0243, -0.0364, 0.1421, 0.0577, 0.1259, 0.0518, -0.0601, -0.0815, 0.0322, 0.0191, -0.1328, 0.0415, -0.1486, 0.1079, 0.1352, -0.0524, 0.105, 0.0338, 0.0029, -0.0293, -0.0792, -0.1452, -0.0799, -0.064, 0.1066, 0.0356, -0.0094, -0.0651, -0.0197, -0.0957, 0.1304, -0.0234, -0.0314, 0.0615, -0.0287, -0.1158, -0.0891, 0.0322, 0.15, -0.0359, -0.0467, -0.0645, -0.1174, 0.0639, -0.0515]))

#### Create a search query in Spark

In [ ]:
SEARCH_QUERY = "coffee shop"
SEARCH_QUERY

'coffee shop'

#### Create a Spark Dataframe for the query

In [ ]:
query_df = spark.createDataFrame([(1, SEARCH_QUERY)]).toDF('index','inputText')

In [ ]:
query_df.show()

+-----+-----------+
|index|  inputText|
+-----+-----------+
|    1|coffee shop|
+-----+-----------+



#### Transform the search query

In [ ]:
query_tok = regextok.transform(query_df)
query_swr = stopwrmv.transform(query_tok)
query_vec = model.transform(query_swr)
query_vec.show()

+-----+-----------+--------------+-----------------+--------------------+
|index|  inputText|        tokens|tokens_sw_removed|         wordvectors|
+-----+-----------+--------------+-----------------+--------------------+
|    1|coffee shop|[coffee, shop]|   [coffee, shop]|[-0.2434652708470...|
+-----+-----------+--------------+-----------------+--------------------+



#### Extract the word vector for the search query

In [ ]:
query_vec = query_vec.select('wordvectors').collect()[0][0]
query_vec

DenseVector([-0.2435, 0.1894, 0.0664, 0.0515, -0.0679, 0.1848, -0.0506, -0.0889, -0.0619, 0.1257, -0.0101, -0.1349, -0.0898, 0.1109, -0.3749, -0.0138, 0.0384, -0.1372, 0.0656, -0.361, -0.3515, -0.0408, 0.0412, 0.5349, -0.1696, -0.2121, 0.0993, -0.2371, 0.117, -0.1464, 0.3829, -0.0916, -0.1045, -0.1559, -0.4286, -0.2226, 0.1432, -0.2881, 0.3986, 0.1448, -0.1463, 0.1103, -0.2388, -0.275, -0.2719, 0.3161, 0.4832, 0.283, 0.2175, 0.1215, -0.0752, -0.0977, -0.1739, 0.1284, 0.1223, 0.2004, 0.2315, 0.0895, 0.1129, 0.1412, -0.0695, 0.2529, -0.0674, -0.1299, -0.0696, 0.0932, -0.205, 0.2312, -0.1544, 0.1578, 0.2697, 0.0275, -0.0302, 0.1255, -0.1954, 0.1443, -0.2672, 0.094, -0.3388, -0.0153, -0.2637, 0.1078, -0.1256, 0.4668, 0.1423, 0.0205, 0.3384, 0.6602, 0.006, 0.116, 0.1201, -0.1968, -0.2336, 0.146, 0.0943, 0.2745, 0.009, 0.1531, -0.1609, -0.1103])

#### Calculate Cosine Similarity

In [ ]:
chunks[:3]

[Row(crunchbase_uuid='e1393508-30ea-8a36-3f96-dd3226033abd', name='Wetpaint', wordvectors=DenseVector([0.1023, 0.0621, -0.051, 0.1017, -0.0687, -0.0639, -0.1182, -0.0996, -0.0093, -0.0358, 0.0208, -0.0773, 0.1145, 0.1586, -0.1143, -0.133, -0.0027, 0.0486, 0.0359, 0.0357, -0.0306, -0.1151, 0.0167, -0.0205, -0.0992, 0.0278, 0.066, 0.0148, 0.0447, -0.1047, 0.1446, -0.0399, 0.0164, -0.0747, -0.1977, 0.0114, -0.02, -0.0285, -0.063, 0.1241, -0.1463, 0.1822, 0.0687, -0.1114, -0.0137, -0.198, 0.0667, 0.0915, 0.0491, -0.2335, 0.1199, -0.0297, -0.0037, 0.0819, -0.0908, -0.0243, -0.0364, 0.1421, 0.0577, 0.1259, 0.0518, -0.0601, -0.0815, 0.0322, 0.0191, -0.1328, 0.0415, -0.1486, 0.1079, 0.1352, -0.0524, 0.105, 0.0338, 0.0029, -0.0293, -0.0792, -0.1452, -0.0799, -0.064, 0.1066, 0.0356, -0.0094, -0.0651, -0.0197, -0.0957, 0.1304, -0.0234, -0.0314, 0.0615, -0.0287, -0.1158, -0.0891, 0.0322, 0.15, -0.0359, -0.0467, -0.0645, -0.1174, 0.0639, -0.0515])),
 Row(crunchbase_uuid='bf4d7b0e-b34d-2fd8-d292-604

In [ ]:
import numpy as np
def cossim(v1, v2):
    '''
        cossim(v1, v2) calculates the cosine similarity between v1 and v1.
        If v1 or v2 is a zero vector, it will return 0
    '''
    if np.dot(v1, v1) == 0 or np.dot(v2, v2) == 0:
        return 0.0
    return float(np.dot(v1, v2) / np.sqrt(np.dot(v1, v1)) / (np.sqrt(np.dot(v2, v2))))

In [ ]:
data = [(i[0], i[1], float(cossim(query_vec, i[2]))) for i in chunks]

In [ ]:
sim_df = spark.createDataFrame(data).toDF('crunchbase_uuid', 'name', 'similarity').orderBy('similarity', ascending=False)

In [ ]:
sim_df.show(20, truncate = False)

+------------------------------------+-------------------------------------------+------------------+
|crunchbase_uuid                     |name                                       |similarity        |
+------------------------------------+-------------------------------------------+------------------+
|68dd3433-e24d-1885-e987-d769d8ab9b21|Starbucks                                  |0.7627639558026453|
|7ba219cf-8227-f8e8-8a88-bbae17ae1eb3|Alice.com                                  |0.7363802891551228|
|c24c8230-4290-8e56-cad0-c8b6f1369c15|Dunkin' Donuts                             |0.7244456243076947|
|f79d2a9b-bd8c-844d-bde0-5286601c3a8d|Mode Flowers                               |0.6848916057325536|
|ecea06bb-9e53-42f7-f4b5-981ee6c3f9c9|GourmetFoodMall.com                        |0.6825377017976145|
|5579a881-3ed5-5d88-d85e-c8f6f3d2c741|Mondo Food                                 |0.677438527942907 |
|45c4e2b6-567a-dc7c-a050-df62bf2fcf0d|1-800-FLOWERS                              |